# Section 8. Retriever를 설계하고 검색 품질 점검하기

공개 실습용 notebook입니다. 네트워크와 API 없이 결정론적으로 실행됩니다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore


def find_data() -> Path:
    candidates = [
        Path("../data/section07_retrieval_corpus.json"),
        Path(__file__).resolve().parents[1] / "data" / "section07_retrieval_corpus.json"
        if "__file__" in globals() else Path("__missing__"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("section07_retrieval_corpus.json을 찾지 못했습니다.")


ALIASES = {
    "비밀": "credential", "인증정보": "credential", "키": "credential", "key": "credential",
    "token": "credential", "유출": "exposure", "노출": "exposure", "공개": "exposure",
    "폐기": "revoke", "재발급": "revoke", "오류": "error", "예외": "error",
    "에러": "error", "질문": "report", "공유": "report", "검색": "retrieval",
    "근거": "evidence", "인용": "evidence", "추측": "unanswerable", "답이": "unanswerable",
    "환경": "environment", "설치": "environment", "python": "python", "uv": "python",
    "kernel": "python", "평가": "evaluation", "순위": "evaluation", "결과": "evaluation",
    "기록": "evaluation",
}


class ConceptEmbedding(Embeddings):
    def __init__(self, dimensions: int = 512): self.dimensions = dimensions
    def _embed(self, text: str) -> list[float]:
        vector = [0.0] * self.dimensions
        normalized = text.lower()
        for surface, concept in ALIASES.items():
            normalized = normalized.replace(surface, f" {concept} ")
        for token in re.findall(r"[0-9A-Za-z가-힣_]+", normalized):
            concept = ALIASES.get(token, token)
            bucket = int.from_bytes(hashlib.sha256(concept.encode()).digest()[:4], "big") % self.dimensions
            vector[bucket] += 1.0
        norm = math.sqrt(sum(v * v for v in vector)) or 1.0
        return [v / norm for v in vector]
    def embed_documents(self, texts: list[str]) -> list[list[float]]: return [self._embed(t) for t in texts]
    def embed_query(self, text: str) -> list[float]: return self._embed(text)


records = json.loads(find_data().read_text(encoding="utf-8"))
documents = [Document(page_content=r["text"], metadata={
    "source_id": r["source_id"], "chunk_id": f"{r['source_id']}:000",
    "title": r["title"], "category": r["category"],
}) for r in records]
store = InMemoryVectorStore(embedding=ConceptEmbedding())
store.add_documents(documents)

## 작은 gold evaluation set

`expected_source=None`은 corpus로 답할 수 없는 질문입니다. 이 질문은 Recall 계산의 분모에서 제외하고,
threshold가 근거 없는 답변을 막을 수 있는지 별도로 봅니다.

In [ ]:
CASES = [
    {"id": "q01", "question": "API key를 공개 저장소에 올려도 되나요?", "expected_source": "security-guide"},
    {"id": "q02", "question": "비밀 인증정보 유출이 의심되면 무엇을 해야 하나요?", "expected_source": "security-guide"},
    {"id": "q03", "question": "에러 질문에는 어떤 정보를 포함하나요?", "expected_source": "debug-guide"},
    {"id": "q04", "question": "개인 경로를 포함한 오류를 공유할 때 주의점은?", "expected_source": "debug-guide"},
    {"id": "q05", "question": "인용문이 실제 근거에 있는지 어떻게 다루나요?", "expected_source": "rag-policy"},
    {"id": "q06", "question": "검색으로 답을 못 찾으면 추측해도 되나요?", "expected_source": "rag-policy"},
    {"id": "q07", "question": "uv로 Python 버전을 확인하는 이유는?", "expected_source": "environment-guide"},
    {"id": "q08", "question": "VS Code에서 package가 다르게 보이면 무엇을 확인하나요?", "expected_source": "environment-guide"},
    {"id": "q09", "question": "검색 설정 변경 전후를 어떤 자료로 비교하나요?", "expected_source": "evaluation-guide"},
    {"id": "q10", "question": "모델 출력 평가에서 입력과 예상 결과를 남겨야 하나요?", "expected_source": "evaluation-guide"},
    {"id": "q11", "question": "연구실 정기 회의는 언제인가요?", "expected_source": None},
    {"id": "q12", "question": "담당 교수님의 전화번호는 무엇인가요?", "expected_source": None},
]


def retrieve(question: str, k: int, *, category: str | None = None) -> list[tuple[Document, float]]:
    # InMemoryVectorStore의 filter callable은 Document를 받습니다.
    filter_fn = None if category is None else lambda doc: doc.metadata.get("category") == category
    return store.similarity_search_with_score(question, k=k, filter=filter_fn)


def recall_at_k(k: int) -> float:
    answerable = [case for case in CASES if case["expected_source"] is not None]
    hits = 0
    for case in answerable:
        sources = {doc.metadata["source_id"] for doc, _ in retrieve(case["question"], k)}
        hits += case["expected_source"] in sources
    return hits / len(answerable)


for k in (1, 3):
    value = recall_at_k(k)
    print(f"Recall@{k}: {value:.3f}")
    assert 0.0 <= value <= 1.0
assert recall_at_k(3) >= recall_at_k(1)

## Metadata filter

filter는 검색 공간을 줄일 수 있지만 잘못된 category를 적용하면 정답 source를 제거합니다.

In [ ]:
question = "에러 질문에는 어떤 정보를 포함하나요?"
unfiltered = retrieve(question, 3)
right_filter = retrieve(question, 3, category="development")
wrong_filter = retrieve(question, 3, category="security")
assert any(d.metadata["source_id"] == "debug-guide" for d, _ in right_filter)
assert all(d.metadata["source_id"] != "debug-guide" for d, _ in wrong_filter)
print("filter sizes:", len(unfiltered), len(right_filter), len(wrong_filter))

## Threshold 정책 탐색

threshold는 현재 corpus와 모델의 평가 결과로 정해야 합니다. 아래 값은 정답이 아니라 관찰 시작점입니다.

In [ ]:
THRESHOLD = 0.30
for case in CASES:
    results = retrieve(case["question"], 1)
    top_source = results[0][0].metadata["source_id"] if results else None
    top_score = results[0][1] if results else float("-inf")
    accepted = top_score >= THRESHOLD
    print(case["id"], "expected=", case["expected_source"], "top=", top_source,
          "score=", round(top_score, 3), "accepted=", accepted)

## 실패 분류

각 miss를 다음 중 하나로 기록합니다.
- query: 질문 표현이나 의도가 모호함
- chunk: 필요한 문맥이 경계에서 분리됨
- corpus: 정답 문서가 없음
- configuration: k, filter, threshold가 정답 후보를 제거함
- representation: 사용한 embedding이 의미 관계를 충분히 표현하지 못함

In [ ]:
answerable = [c for c in CASES if c["expected_source"]]
misses = []
for case in answerable:
    top = retrieve(case["question"], 1)[0][0].metadata["source_id"]
    if top != case["expected_source"]:
        misses.append((case["id"], case["expected_source"], top, "representation/query 확인"))
print("top-1 misses:", misses)

assert len(CASES) == 12
assert sum(c["expected_source"] is None for c in CASES) == 2
print("evaluation set structure: PASS")

## 제출물

1. Recall@1과 Recall@3 및 차이의 이유
2. 올바른 filter와 잘못된 filter의 결과
3. 답 없음 두 질문의 top score와 threshold 판정
4. 실패 사례 2개를 원인 범주와 함께 기록